# CompactLLM — QLoRA fine-tune on Colab

Trains the Gemma-3 4B resume/JD relevance scorer, evaluates it against the base
model, and packages the adapter + benchmark for download.

> **Colab free tier caps GPU at ~3–4h/day and disconnects when you hit it.**
> This run is ~2h train + ~40min eval — cutting it close. If you hit a
> "compute units" prompt or a disconnect, use `training/train_kaggle.ipynb`
> instead (30 GPU-hrs/week, far more reliable).

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Accept the Gemma license: https://huggingface.co/google/gemma-3-4b-it
3. Add Colab secrets (🔑 in the sidebar): `HF_TOKEN` (required, read scope),
   `WANDB_API_KEY` (optional — loss curves), `GROQ_API_KEY` (optional — the
   eval rationale judge)

Then: Runtime → Run all.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone --depth 1 https://github.com/aayush-arya/compact-llm.git
%cd compact-llm

In [ ]:
# Unsloth's pip package pulls a Colab-compatible torch/trl/peft stack.
# If it ever conflicts, fall back to: !pip install -q -r training/requirements.txt
!pip install -q unsloth
!pip install -q wandb httpx

In [ ]:
# The dataset ships in the repo now, so this is just a sanity check.
!wc -l data/processed/*.jsonl

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
for opt in ('WANDB_API_KEY', 'GROQ_API_KEY', 'CEREBRAS_API_KEY', 'GEMINI_API_KEY'):
    try:
        os.environ[opt] = userdata.get(opt)
    except Exception:
        pass

In [ ]:
# Writes outputs/adapter/ and outputs/merged/. ~2h on a T4 (Gemma-3 forces
# float32 there); ~30-45 min on an A100/L4. Keep the tab active -- Colab
# disconnects idle sessions and this run doesn't auto-resume.
!python training/train_unsloth_qlora.py

In [ ]:
# Held-out eval: base zero-shot vs few-shot vs fine-tuned. ~35-40 min.
# --judge auto uses GROQ/CEREBRAS/GEMINI_API_KEY if set, else skips rationale grading.
!python training/eval_base_vs_finetuned.py --judge auto

In [ ]:
# Package just the adapter (~120 MB) + benchmark. The merged fp16 model in
# outputs/merged is ~8 GB and only needed for the GGUF/Ollama export path.
!zip -r compactllm-outputs.zip outputs/adapter docs/benchmark_results.json docs/benchmark_table.md
from google.colab import files
files.download('compactllm-outputs.zip')

## Back on your machine

```bash
unzip compactllm-outputs.zip   # -> outputs/adapter/, docs/benchmark_results.json
```

`outputs/adapter/` is git-ignored — push it to a Hugging Face model repo for
deployment. `docs/benchmark_results.json` is committed: it lights up the
Evaluation page and the README table. Then run the app with
`MODEL_BACKEND=transformers`.

Need the merged fp16 weights too (for a GGUF/Ollama export)? Add a cell:
`!zip -r merged.zip outputs/merged && files.download('merged.zip')`.